# LeetCode 3340 — Check Balanced String bằng Python

Link đề: https://leetcode.com/problems/check-balanced-string/ · Độ khó: **Easy** · Chủ đề: String, Simulation, Math

Skeleton LeetCode:

```python
class Solution:
    def isBalanced(self, num: str) -> bool:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 422); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `num` chỉ gồm các chữ số; nó **cân bằng** nếu tổng các chữ số ở **chỉ số chẵn** bằng tổng các chữ số ở **chỉ số lẻ** (chỉ số tính từ `0`).
2. Nắm các công cụ Python: chuyển ký tự số thành số nguyên bằng `int(ch)`, `enumerate` để vừa có chỉ số vừa có ký tự, kiểm tra chẵn/lẻ bằng `i % 2` và `i & 1`, **cắt lát có bước** `num[0::2]` / `num[1::2]`, và `sum(map(int, ...))`.
3. Đi từ cách **hai biến tổng** → **cắt lát theo bước** → **một biến hiệu** (`chẵn − lẻ`) → **mảng hai ô `f[i & 1]`** → **đổi dấu luân phiên**.
4. Hiểu mấu chốt: ta chỉ cần biết hai tổng có **bằng nhau** hay không, nên gói lại thành **một con số duy nhất** là hiệu `tổng chẵn − tổng lẻ` và kiểm tra nó bằng `0`.
5. Tránh các lỗi kinh điển: nhầm **chỉ số chẵn** với **chữ số chẵn**, dùng `ord(ch)` mà quên trừ mã của `'0'` (chỉ đúng khi độ dài chẵn), cộng chuỗi mà quên `int(...)`, và bỏ sót chữ số cuối do `range(len(num) - 1)`.

## 2. Đề bài & ví dụ

Cho chuỗi `num` chỉ gồm các chữ số. Một chuỗi chữ số được gọi là **cân bằng** nếu **tổng các chữ số ở chỉ số chẵn** bằng **tổng các chữ số ở chỉ số lẻ** (chỉ số đánh từ `0`).

Trả về `true` nếu `num` cân bằng, ngược lại trả về `false`.

**Ví dụ 1**

- Input: `num = "1234"`
- Output: `false`
- Giải thích: tổng ở chỉ số chẵn là `1 + 3 = 4`, tổng ở chỉ số lẻ là `2 + 4 = 6`. `4 != 6` nên không cân bằng.

**Ví dụ 2**

- Input: `num = "24123"`
- Output: `true`
- Giải thích: chỉ số chẵn: `2 + 1 + 3 = 6`; chỉ số lẻ: `4 + 2 = 6`. Hai tổng bằng nhau nên cân bằng.

**Ràng buộc:** `2 <= num.length <= 100`; `num` chỉ gồm các chữ số.

In [1]:
import random

examples = [
    ("1234", False),
    ("24123", True),
]
for num, expected in examples:
    print(repr(num), "-> kỳ vọng", expected)

'1234' -> kỳ vọng False
'24123' -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Ký tự số và số nguyên: `int(ch)`

Mỗi phần tử `num[i]` là một **ký tự** (chuỗi dài 1), ví dụ `"7"`, **không phải** số `7`. Nếu cộng trực tiếp các ký tự thì ta nối chuỗi chứ không cộng số. Hàm `int("7")` đổi ký tự thành số nguyên `7`.

In [2]:
ch = "7"
print(type(ch), type(int(ch)))           # <class 'str'> <class 'int'>
print("7" + "5", int("7") + int("5"))    # 75 12   (nối chuỗi khác cộng số!)

<class 'str'> <class 'int'>
75 12


### 3.2. Chỉ số chẵn / lẻ: `i % 2` và `i & 1`

`i % 2` là phần dư khi chia `i` cho 2: `0` nếu `i` chẵn, `1` nếu `i` lẻ. Phép `i & 1` (AND bit với 1) cho kết quả y hệt vì bit thấp nhất của `i` cho biết chẵn/lẻ. Nhớ: ta xét **chỉ số** `i` chứ **không phải** giá trị chữ số.

In [3]:
for i in range(6):
    print(i, i % 2, i & 1, "chẵn" if i % 2 == 0 else "lẻ")

0 0 0 chẵn
1 1 1 lẻ
2 0 0 chẵn
3 1 1 lẻ
4 0 0 chẵn
5 1 1 lẻ


### 3.3. `enumerate`: vừa có chỉ số vừa có phần tử

`for i, ch in enumerate(s)` cho từng cặp `(chỉ_số, ký_tự)`, thay cho cách viết `for i in range(len(s))` rồi `s[i]`.

In [4]:
for i, ch in enumerate("2413"):
    print(f"chỉ số {i} -> chữ số {ch} ({'chẵn' if i % 2 == 0 else 'lẻ'})")

chỉ số 0 -> chữ số 2 (chẵn)
chỉ số 1 -> chữ số 4 (lẻ)
chỉ số 2 -> chữ số 1 (chẵn)
chỉ số 3 -> chữ số 3 (lẻ)


### 3.4. Cắt lát có bước: `s[start::step]`

`s[0::2]` lấy các phần tử ở chỉ số `0, 2, 4, ...` (chỉ số **chẵn**); `s[1::2]` lấy `1, 3, 5, ...` (chỉ số **lẻ**). Cú pháp là `s[bắt_đầu:kết_thúc:bước]`; bỏ trống `kết_thúc` nghĩa là tới hết chuỗi.

In [5]:
s = "abcde"
print(s[0::2], s[1::2])                  # ace bd
num = "24123"
print(num[0::2], num[1::2])              # 213 42  (chỉ số chẵn / chỉ số lẻ)

ace bd
213 42


### 3.5. `map` và `sum`

`map(int, "213")` áp `int` lên từng ký tự của `"213"` (cho `2, 1, 3`); `sum(...)` cộng tất cả. Gộp lại: `sum(map(int, "213")) = 6`. Một chuỗi rỗng cho tổng `0` (không lỗi), điều này hữu ích khi chuỗi chỉ có 1 phần tử ở một vị trí.

In [6]:
print(sum(map(int, "213")))              # 6
print(sum(map(int, "")))                 # 0
print(list(map(int, "24123")))           # [2, 4, 1, 2, 3]
print(sum(int(c) for c in "42"))         # 6  (cách viết tương đương bằng generator)

6
0
[2, 4, 1, 2, 3]
6


### 3.6. Trả về kết quả so sánh trực tiếp

Biểu thức `a == b` đã là `True`/`False`, nên ta viết thẳng `return a == b` thay vì `if a == b: return True else: return False`.

In [7]:
a, b = 6, 6
print(a == b)                            # True
print(4 == 6)                            # False

True
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `num = "24123"`. Viết các chữ số kèm chỉ số:

| chỉ số | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| chữ số | 2 | 4 | 1 | 2 | 3 |
| chẵn/lẻ | chẵn | lẻ | chẵn | lẻ | chẵn |

- Tổng chỉ số **chẵn**: `2 + 1 + 3 = 6`.
- Tổng chỉ số **lẻ**: `4 + 2 = 6`.
- Hai tổng bằng nhau ⇒ **cân bằng** (`True`).

Với `"1234"`: chẵn `1 + 3 = 4`, lẻ `2 + 4 = 6` ⇒ `False`.

Lưu ý quan trọng: "chẵn/lẻ" ở đây là của **vị trí** (chỉ số), không phải của **chữ số**. Ở `"24123"` chữ số `2` ở chỉ số `0` (chẵn), còn chữ số `2` ở chỉ số `3` (lẻ).

In [8]:
num = "24123"
even_digits = [int(num[i]) for i in range(len(num)) if i % 2 == 0]
odd_digits = [int(num[i]) for i in range(len(num)) if i % 2 == 1]
print("chỉ số chẵn:", even_digits, "tổng", sum(even_digits))   # [2, 1, 3] tổng 6
print("chỉ số lẻ:", odd_digits, "tổng", sum(odd_digits))       # [4, 2] tổng 6
print("cân bằng?", sum(even_digits) == sum(odd_digits))        # True

chỉ số chẵn: [2, 1, 3] tổng 6
chỉ số lẻ: [4, 2] tổng 6
cân bằng? True


## 5. Cách 1 — Brute Force / baseline: hai biến tổng

**Ý tưởng.** Dùng hai biến `even_sum` và `odd_sum`. Duyệt từng chữ số với chỉ số `i`: nếu `i` chẵn cộng vào `even_sum`, ngược lại cộng vào `odd_sum`. Cuối cùng so sánh hai biến.

In [9]:
def is_balanced_two_sums(num: str) -> bool:
    even_sum = 0
    odd_sum = 0
    for i, ch in enumerate(num):
        digit = int(ch)                  # đổi ký tự thành số
        if i % 2 == 0:
            even_sum += digit            # chỉ số chẵn
        else:
            odd_sum += digit             # chỉ số lẻ
    return even_sum == odd_sum


for num, expected in examples:
    print(is_balanced_two_sums(num), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** Duyệt một lần `n` chữ số ⇒ **Time `O(n)`**, **Space `O(1)`**.

Cách này đã tối ưu về bậc; các cách sau chỉ là **viết gọn hơn** hoặc **nhìn khác đi**. Tuy vậy ta vẫn có điều nên cải tiến: ta đang giữ **hai** biến trong khi đề chỉ hỏi chúng có bằng nhau hay không.

## 6. Ý tưởng thuật toán / bước đột phá

Hai tổng `E` (chẵn) và `O` (lẻ) bằng nhau **khi và chỉ khi** `E − O = 0`. Vậy ta không cần nhớ riêng từng tổng; chỉ cần **một biến hiệu** `diff = E − O`:

- gặp chữ số ở chỉ số **chẵn** ⇒ `diff += digit`;
- gặp chữ số ở chỉ số **lẻ** ⇒ `diff -= digit`;
- cuối cùng `diff == 0` là cân bằng.

Cùng một ý, ta có thể xem đây là **tổng đan dấu** `d0 − d1 + d2 − d3 + ...`. Bài này chính là "tổng đan dấu bằng 0?". Một nhận xét phụ: nếu **tổng cả chuỗi lẻ** thì chắc chắn **không** cân bằng (vì `E + O = 2E` luôn chẵn khi `E = O`) — có thể dùng để loại nhanh.

In [10]:
for num in ["1234", "24123", "00", "5"]:
    diff = 0
    for i, ch in enumerate(num):
        diff += int(ch) if i % 2 == 0 else -int(ch)
    total = sum(map(int, num))
    print(f"{num!r:8} diff = {diff:3d}, tổng cả chuỗi = {total}, tổng lẻ -> chắc chắn False? {total % 2 == 1}")

'1234'   diff =  -2, tổng cả chuỗi = 10, tổng lẻ -> chắc chắn False? False
'24123'  diff =   0, tổng cả chuỗi = 12, tổng lẻ -> chắc chắn False? False
'00'     diff =   0, tổng cả chuỗi = 0, tổng lẻ -> chắc chắn False? False
'5'      diff =   5, tổng cả chuỗi = 5, tổng lẻ -> chắc chắn False? True


## 7. Các cách tiếp theo

### Cách 2 — Cắt lát theo bước

**Giải thích từng đoạn.**

1. `num[0::2]` là chuỗi các chữ số ở chỉ số **chẵn**; `num[1::2]` là các chữ số ở chỉ số **lẻ**.
2. `sum(map(int, ...))` đổi từng ký tự thành số rồi cộng.
3. So sánh hai tổng bằng `==`.

Đây là cách ngắn nhất và gần với mô tả đề nhất.

In [11]:
def is_balanced_slices(num: str) -> bool:
    even_sum = sum(map(int, num[0::2]))  # các chữ số ở chỉ số chẵn
    odd_sum = sum(map(int, num[1::2]))   # các chữ số ở chỉ số lẻ
    return even_sum == odd_sum


for num, expected in examples:
    print(is_balanced_slices(num), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
True (kỳ vọng True )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSlices:
    def isBalanced(self, num: str) -> bool:
        return sum(map(int, num[0::2])) == sum(map(int, num[1::2]))


print(SolutionSlices().isBalanced("1234"), SolutionSlices().isBalanced("24123"))   # False True

False True


**Độ phức tạp Cách 2:** cắt lát tạo hai chuỗi tổng dài `n` ⇒ **Time `O(n)`**, **Space `O(n)`** (hai chuỗi con).

### Cách 3 — Một biến hiệu `diff`

**Giải thích từng đoạn.** Áp dụng đúng bước đột phá mục 6:

1. `diff = 0`.
2. Duyệt `i, ch`: chỉ số chẵn cộng `digit`, lẻ trừ `digit`.
3. `return diff == 0`.

In [13]:
def is_balanced_diff(num: str) -> bool:
    diff = 0                             # diff = (tổng chẵn) - (tổng lẻ)
    for i, ch in enumerate(num):
        if i % 2 == 0:
            diff += int(ch)
        else:
            diff -= int(ch)
    return diff == 0


for num, expected in examples:
    print(is_balanced_diff(num), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
True (kỳ vọng True )


In [14]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionDiff:
    def isBalanced(self, num: str) -> bool:
        diff = 0
        for i, ch in enumerate(num):
            diff += int(ch) if i % 2 == 0 else -int(ch)
        return diff == 0


print(SolutionDiff().isBalanced("1234"), SolutionDiff().isBalanced("24123"))       # False True

False True


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Mảng hai ô `f[i & 1]`

**Giải thích.** Thay `if/else` bằng một **mảng hai phần tử** `f = [0, 0]`: `f[0]` giữ tổng chỉ số chẵn, `f[1]` giữ tổng chỉ số lẻ. Với chỉ số `i`, ô cần cộng là `i & 1` (hoặc `i % 2`) vì nó cho `0` khi chẵn và `1` khi lẻ. Đây là mẹo "dùng chính chẵn/lẻ làm chỉ số mảng", gọn và không cần nhánh.

In [15]:
def is_balanced_array(num: str) -> bool:
    f = [0, 0]                           # f[0]: tổng chỉ số chẵn, f[1]: tổng chỉ số lẻ
    for i, ch in enumerate(num):
        f[i & 1] += int(ch)              # i & 1 = 0 nếu i chẵn, 1 nếu i lẻ
    return f[0] == f[1]


for num, expected in examples:
    print(is_balanced_array(num), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
True (kỳ vọng True )


In [16]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionArray:
    def isBalanced(self, num: str) -> bool:
        f = [0, 0]
        for i, x in enumerate(map(int, num)):
            f[i & 1] += x
        return f[0] == f[1]


print(SolutionArray().isBalanced("1234"), SolutionArray().isBalanced("24123"))     # False True

False True


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (mảng 2 ô).

### Cách 5 — Dấu luân phiên `(-1) ** i`

**Giải thích.** Chỉ số chẵn ứng dấu `+`, lẻ ứng dấu `-`: đó đúng là `(-1) ** i` (`(-1)^0 = 1`, `(-1)^1 = -1`, `(-1)^2 = 1`, ...). Vậy hiệu `E − O` là `sum((-1) ** i * int(ch) for i, ch in enumerate(num))`, và ta so với `0`. Gọn nhưng lũy thừa chậm hơn một chút so với `if`; với `n <= 100` không đáng kể.

In [17]:
print([(-1) ** i for i in range(6)])     # [1, -1, 1, -1, 1, -1]


def is_balanced_alternating(num: str) -> bool:
    return sum((-1) ** i * int(ch) for i, ch in enumerate(num)) == 0


for num, expected in examples:
    print(is_balanced_alternating(num), "(kỳ vọng", expected, ")")

[1, -1, 1, -1, 1, -1]
False (kỳ vọng False )
True (kỳ vọng True )


In [18]:
# Code nộp LeetCode cho Cách 5 (đổi tên lớp thành Solution khi nộp)
class SolutionAlternating:
    def isBalanced(self, num: str) -> bool:
        return sum((-1) ** i * int(ch) for i, ch in enumerate(num)) == 0


print(SolutionAlternating().isBalanced("24123"))                                   # True

True


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

**Bảng trace Cách 3** (một biến hiệu) với `num = "24123"`:

| i | chữ số | chẵn/lẻ | thao tác | `diff` sau bước |
|---|---|---|---|---|
| (bắt đầu) | | | | 0 |
| 0 | 2 | chẵn | `+2` | 2 |
| 1 | 4 | lẻ | `-4` | -2 |
| 2 | 1 | chẵn | `+1` | -1 |
| 3 | 2 | lẻ | `-2` | -3 |
| 4 | 3 | chẵn | `+3` | 0 |

`diff = 0` ⇒ cân bằng. Ở Cách 1, cùng các bước đó nhưng tách thành `even_sum = 6`, `odd_sum = 6`.

In [19]:
def trace_two_sums(num: str) -> bool:
    even_sum = odd_sum = 0
    print(f"num = {num!r}")
    for i, ch in enumerate(num):
        digit = int(ch)
        if i % 2 == 0:
            even_sum += digit
            where = "chẵn"
        else:
            odd_sum += digit
            where = "lẻ  "
        print(f"  i={i}, chữ số={digit}, chỉ số {where} -> even_sum={even_sum}, odd_sum={odd_sum}")
    result = even_sum == odd_sum
    print(f"  {even_sum} == {odd_sum} ? {result}")
    return result


assert trace_two_sums("24123") is True
assert trace_two_sums("1234") is False

num = '24123'
  i=0, chữ số=2, chỉ số chẵn -> even_sum=2, odd_sum=0
  i=1, chữ số=4, chỉ số lẻ   -> even_sum=2, odd_sum=4
  i=2, chữ số=1, chỉ số chẵn -> even_sum=3, odd_sum=4
  i=3, chữ số=2, chỉ số lẻ   -> even_sum=3, odd_sum=6
  i=4, chữ số=3, chỉ số chẵn -> even_sum=6, odd_sum=6
  6 == 6 ? True
num = '1234'
  i=0, chữ số=1, chỉ số chẵn -> even_sum=1, odd_sum=0
  i=1, chữ số=2, chỉ số lẻ   -> even_sum=1, odd_sum=2
  i=2, chữ số=3, chỉ số chẵn -> even_sum=4, odd_sum=2
  i=3, chữ số=4, chỉ số lẻ   -> even_sum=4, odd_sum=6
  4 == 6 ? False


In [20]:
def trace_diff(num: str) -> bool:
    diff = 0
    print(f"num = {num!r}")
    for i, ch in enumerate(num):
        digit = int(ch)
        if i % 2 == 0:
            diff += digit
            op = f"+{digit}"
        else:
            diff -= digit
            op = f"-{digit}"
        print(f"  i={i}: {op:>3} -> diff={diff}")
    print("  diff == 0 ?", diff == 0)
    return diff == 0


assert trace_diff("24123") is True
assert trace_diff("1234") is False

num = '24123'
  i=0:  +2 -> diff=2
  i=1:  -4 -> diff=-2
  i=2:  +1 -> diff=-1
  i=3:  -2 -> diff=-3
  i=4:  +3 -> diff=0
  diff == 0 ? True
num = '1234'
  i=0:  +1 -> diff=1
  i=1:  -2 -> diff=-1
  i=2:  +3 -> diff=2
  i=3:  -4 -> diff=-2
  diff == 0 ? False


In [21]:
def trace_slices(num: str) -> bool:
    even_part, odd_part = num[0::2], num[1::2]
    even_sum, odd_sum = sum(map(int, even_part)), sum(map(int, odd_part))
    print(f"num = {num!r}: chỉ số chẵn -> {even_part!r} (tổng {even_sum}), chỉ số lẻ -> {odd_part!r} (tổng {odd_sum})")
    return even_sum == odd_sum


assert trace_slices("24123") is True
assert trace_slices("1234") is False

num = '24123': chỉ số chẵn -> '213' (tổng 6), chỉ số lẻ -> '42' (tổng 6)
num = '1234': chỉ số chẵn -> '13' (tổng 4), chỉ số lẻ -> '24' (tổng 6)


## 9. Code nộp LeetCode

Ta nộp **Cách 3** (một biến hiệu): rõ ràng, `O(n)` thời gian và `O(1)` bộ nhớ, và nó thể hiện đúng ý "tổng đan dấu bằng 0".

In [22]:
class Solution:
    def isBalanced(self, num: str) -> bool:
        diff = 0                             # diff = (tổng chỉ số chẵn) - (tổng chỉ số lẻ)
        for i, ch in enumerate(num):
            if i % 2 == 0:
                diff += int(ch)
            else:
                diff -= int(ch)
        return diff == 0


sol = Solution()
print(sol.isBalanced("1234"), sol.isBalanced("24123"))     # False True

False True


## 10. Vì sao đúng

- **Mỗi chữ số được tính đúng một lần.** Vòng lặp đi qua mọi chỉ số `0..n-1` đúng một lần; chữ số ở chỉ số chẵn được cộng, ở chỉ số lẻ bị trừ.
- **Biến hiệu mã hoá đúng điều kiện.** Sau vòng lặp `diff = E − O` với `E` là tổng chỉ số chẵn, `O` là tổng chỉ số lẻ. Đề đòi `E = O`, tức `E − O = 0`.
- **Không phụ thuộc cách đánh chỉ số.** Nếu bạn đánh chỉ số từ `1` thay vì `0`, vai trò chẵn/lẻ hoán đổi, `diff` đổi dấu nhưng `diff == 0` vẫn như cũ — đáp án không đổi.
- **Các cách khác tương đương.** Cách 1 giữ riêng `E` và `O`; Cách 2 tính `E`, `O` bằng cắt lát; Cách 4 dùng `f[0] = E`, `f[1] = O`; Cách 5 tính `E − O` với dấu `(-1)^i`. Tất cả chỉ khác cách biểu diễn.
- **Nhận xét độ dài:** vì `n >= 2` luôn có ít nhất một chữ số ở mỗi loại chỉ số; nhưng ngay cả khi một loại rỗng, tổng của nó là `0` nên code vẫn đúng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(num)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Hai biến tổng | `O(n)` | `O(1)` |
| 2. Cắt lát theo bước | `O(n)` | `O(n)` |
| 3. Một biến hiệu | `O(n)` | `O(1)` |
| 4. Mảng `f[i & 1]` | `O(n)` | `O(1)` |
| 5. Dấu `(-1) ** i` | `O(n)` | `O(1)` |

Bài chỉ cần đọc mỗi chữ số một lần nên không thể tốt hơn `O(n)`.

## 12. Bộ test

Để đối chiếu **độc lập**, ta viết `reference_pairs(num)`: **ghép từng cặp** chữ số liền nhau `(d0, d1), (d2, d3), ...` và cộng dồn `d_chẵn − d_lẻ` của mỗi cặp (chữ số cuối lẻ loi nếu độ dài lẻ chỉ góp phần `+`). Đây là một cách tính khác hẳn các cách ở trên (không dùng `i % 2`, `enumerate` hay cắt lát bước).

In [23]:
def reference_pairs(num):
    total = 0
    k = 0
    while k < len(num):
        first = int(num[k])                                # chữ số ở chỉ số chẵn k
        second = int(num[k + 1]) if k + 1 < len(num) else 0  # chữ số ở chỉ số lẻ k + 1 (nếu có)
        total += first - second
        k += 2
    return total == 0


impls = {
    "two_sums": is_balanced_two_sums,
    "slices": is_balanced_slices,
    "diff": is_balanced_diff,
    "array": is_balanced_array,
    "alternating": is_balanced_alternating,
    "leetcode": Solution().isBalanced,
}


def check_all(num, expected, label):
    assert reference_pairs(num) == expected, ("tham chiếu", label, num)
    for name, fn in impls.items():
        actual = fn(num)
        assert actual == expected, (label, name, num, actual, expected)
    print("PASS", label)


for idx, (num, expected) in enumerate(examples, 1):
    check_all(num, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Độ dài tối thiểu `2`; chuỗi toàn số `0`; hai chữ số bằng nhau; hai chữ số khác nhau; độ dài lẻ; chữ số `9` nhiều; độ dài tối đa `100`.

In [24]:
check_all("00", True, "hai số 0")
check_all("11", True, "hai chữ số bằng nhau")
check_all("12", False, "hai chữ số khác nhau")
check_all("90", False, "9 và 0")
check_all("505", False, "độ dài lẻ: 5+5 = 10 vs 0")
check_all("1331", True, "bài tập 2: 1+3 = 3+1")
check_all("2002", True, "bài tập 2: 2+0 = 0+2")
check_all("9" * 100, True, "100 chữ số 9 (50 chỗ chẵn, 50 chỗ lẻ)")
check_all("9" * 99, False, "99 chữ số 9 (50 chỗ chẵn, 49 chỗ lẻ)")
check_all("0" * 100, True, "toàn 0, dài nhất")
check_all("9" + "0" * 98 + "9", True, "9 ở chỉ số 0 và 9 ở chỉ số 99 (chẵn vs lẻ)")
check_all("9" + "0" * 99, False, "9 chỉ ở chỉ số 0")
check_all("0123456789", False, "đủ 10 chữ số: chẵn 20 vs lẻ 25")
check_all("00000000009", False, "11 ký tự, 9 ở chỉ số chẵn 10")

PASS hai số 0
PASS hai chữ số bằng nhau
PASS hai chữ số khác nhau
PASS 9 và 0
PASS độ dài lẻ: 5+5 = 10 vs 0
PASS bài tập 2: 1+3 = 3+1
PASS bài tập 2: 2+0 = 0+2
PASS 100 chữ số 9 (50 chỗ chẵn, 50 chỗ lẻ)
PASS 99 chữ số 9 (50 chỗ chẵn, 49 chỗ lẻ)
PASS toàn 0, dài nhất
PASS 9 ở chỉ số 0 và 9 ở chỉ số 99 (chẵn vs lẻ)
PASS 9 chỉ ở chỉ số 0
PASS đủ 10 chữ số: chẵn 20 vs lẻ 25
PASS 11 ký tự, 9 ở chỉ số chẵn 10


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi: (a) xét **chữ số** chẵn/lẻ thay vì **chỉ số**; (b) dùng `ord(ch)` mà không trừ `ord("0")` (chỉ đúng khi độ dài chẵn); (c) bỏ sót chữ số cuối do `range(len(num) - 1)`.

In [25]:
def wrong_digit_parity(num):
    even_sum = sum(int(c) for c in num if int(c) % 2 == 0)    # SAI: xét giá trị chữ số
    odd_sum = sum(int(c) for c in num if int(c) % 2 == 1)
    return even_sum == odd_sum


def wrong_ord(num):
    even_sum = sum(ord(c) for c in num[0::2])                  # SAI: ord('0') = 48, không phải 0
    odd_sum = sum(ord(c) for c in num[1::2])
    return even_sum == odd_sum


def wrong_skip_last(num):
    diff = 0
    for i in range(len(num) - 1):                              # SAI: bỏ chữ số cuối
        diff += int(num[i]) if i % 2 == 0 else -int(num[i])
    return diff == 0


tests = [("1234", False), ("24123", True), ("12", False), ("505", False)]
for fn in (wrong_digit_parity, wrong_ord, wrong_skip_last):
    results = [(n, fn(n), e) for n, e in tests]
    print(fn.__name__, results)
    assert any(got != exp for _, got, exp in results), fn.__name__

wrong_digit_parity [('1234', False, False), ('24123', False, True), ('12', False, False), ('505', False, False)]
wrong_ord [('1234', False, False), ('24123', False, True), ('12', False, False), ('505', False, False)]
wrong_skip_last [('1234', False, False), ('24123', False, True), ('12', False, False), ('505', False, False)]


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `num` độ dài `2..100`. Vì chuỗi ngẫu nhiên thuần túy gần như luôn `False`, ta trộn **ba loại**: (1) chữ số hoàn toàn ngẫu nhiên; (2) chuỗi **chắc chắn cân bằng** (chọn tổng mục tiêu rồi chia ngẫu nhiên cho các chỉ số chẵn và lẻ); (3) chuỗi cân bằng bị **đổi một chữ số** (gần cân bằng, hay là phản ví dụ sát). Ta kiểm tra có cả `True` lẫn `False`.

In [26]:
def distribute(rng, total, slots):
    # chia `total` thành `slots` số trong [0, 9] (cần total <= 9 * slots), ngẫu nhiên
    parts = [0] * slots
    for _ in range(total):
        i = rng.randrange(slots)
        while parts[i] == 9:
            i = rng.randrange(slots)
        parts[i] += 1
    return parts


def gen_balanced(rng, n):
    m_even, m_odd = (n + 1) // 2, n // 2
    target = rng.randint(0, 9 * min(m_even, m_odd))
    evens = distribute(rng, target, m_even)
    odds = distribute(rng, target, m_odd)
    digits = []
    for i in range(n):
        digits.append(evens[i // 2] if i % 2 == 0 else odds[i // 2])
    return "".join(map(str, digits))


rng = random.Random(3340)
counts = {True: 0, False: 0}
for trial in range(3000):
    n = rng.randint(2, 100)
    kind = rng.choice(["random", "balanced", "near"])
    if kind == "random":
        num = "".join(rng.choice("0123456789") for _ in range(n))
    elif kind == "balanced":
        num = gen_balanced(rng, n)
    else:
        num = list(gen_balanced(rng, n))
        pos = rng.randrange(n)
        num[pos] = str((int(num[pos]) + rng.randint(1, 9)) % 10)   # đổi một chữ số
        num = "".join(num)
    assert 2 <= len(num) <= 100 and num.isdigit()
    ref = reference_pairs(num)
    counts[ref] += 1
    for name, fn in impls.items():
        assert fn(num) == ref, (name, num, fn(num), ref)
    if kind == "balanced":
        assert ref is True
print("PASS 3000 chuỗi ngẫu nhiên; phân bố kết quả tham chiếu:", counts)
assert counts[True] > 500 and counts[False] > 500

PASS 3000 chuỗi ngẫu nhiên; phân bố kết quả tham chiếu: {True: 1007, False: 1993}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm "chỉ số chẵn" với "chữ số chẵn"

In [27]:
def sai_chu_so_chan(num):
    even_sum = odd_sum = 0
    for ch in num:
        d = int(ch)
        if d % 2 == 0:                   # SAI: xét giá trị chữ số, không phải vị trí
            even_sum += d
        else:
            odd_sum += d
    return even_sum == odd_sum


print("sai:", sai_chu_so_chan("24123"), "| đúng:", Solution().isBalanced("24123"))    # False | True
print("sai:", sai_chu_so_chan("11"), "| đúng:", Solution().isBalanced("11"))          # False | True

sai: False | đúng: True
sai: False | đúng: True


**Vì sao sai:** đề nói **chỉ số chẵn/lẻ** (vị trí trong chuỗi). Code sai phân loại theo **giá trị** chữ số: với `"11"` cả hai chữ số đều lẻ nên `even_sum = 0 ≠ odd_sum = 2`, trong khi đúng ra chúng ở chỉ số `0` và `1` và bằng nhau. **Cách sửa:** điều kiện là `i % 2 == 0` với `i` là chỉ số (dùng `enumerate`).

### Lỗi 2: Dùng `ord(ch)` mà quên trừ mã của `'0'`

In [28]:
def sai_dung_ord(num):
    even_sum = sum(ord(c) for c in num[0::2])
    odd_sum = sum(ord(c) for c in num[1::2])
    return even_sum == odd_sum


print("1234 :", sai_dung_ord("1234"), "| đúng:", Solution().isBalanced("1234"))        # False | False  (độ dài chẵn: trùng hợp đúng)
print("24123:", sai_dung_ord("24123"), "| đúng:", Solution().isBalanced("24123"))      # False | True    (độ dài lẻ: sai)

1234 : False | đúng: False
24123: False | đúng: True


**Vì sao sai:** `ord("0") = 48` nên mỗi chữ số được cộng thêm `48`. Khi độ dài **chẵn**, hai bên có **cùng** số chữ số nên `48 × (số chữ số)` triệt tiêu và kết quả đúng (do trùng hợp). Khi độ dài **lẻ**, bên chẵn nhiều hơn bên lẻ một chữ số ⇒ chênh `48` và sai. **Cách sửa:** dùng `int(ch)` hoặc `ord(ch) - ord("0")`.

### Lỗi 3: Cộng ký tự mà quên đổi sang số

In [29]:
def sai_quen_int(num):
    even_sum = 0
    odd_sum = 0
    for i, ch in enumerate(num):
        if i % 2 == 0:
            even_sum += ch               # SAI: cộng một chuỗi vào số
        else:
            odd_sum += ch
    return even_sum == odd_sum


try:
    sai_quen_int("1234")
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().isBalanced("1234"))

TypeError: unsupported operand type(s) for +=: 'int' and 'str'
đúng: False


**Vì sao sai:** `0 + "1"` không hợp lệ (không cộng được số với chuỗi) nên Python báo `TypeError`. Nếu khởi tạo biến là chuỗi rỗng `""` thì sẽ không lỗi nhưng **nối chuỗi** (`"1" + "3" = "13"`) rồi so sánh các chuỗi nối là sai nghĩa. **Cách sửa:** luôn `int(ch)` trước khi cộng.

### Lỗi 4: Vòng lặp `range(len(num) - 1)` bỏ chữ số cuối

In [30]:
def sai_bo_chu_so_cuoi(num):
    diff = 0
    for i in range(len(num) - 1):        # SAI: không xét chữ số cuối cùng
        diff += int(num[i]) if i % 2 == 0 else -int(num[i])
    return diff == 0


print("sai:", sai_bo_chu_so_cuoi("24123"), "| đúng:", Solution().isBalanced("24123"))   # False | True
print("sai:", sai_bo_chu_so_cuoi("12"), "| đúng:", Solution().isBalanced("12"))         # False | False
print("sai:", sai_bo_chu_so_cuoi("10"), "| đúng:", Solution().isBalanced("10"))         # False | False

sai: False | đúng: True
sai: False | đúng: False
sai: False | đúng: False


**Vì sao sai:** `range(len(num) - 1)` đi tới chỉ số `n - 2` và **bỏ** chỉ số `n - 1`. Với `"24123"` thiếu chữ số `3` ở cuối nên `diff = 2 − 4 + 1 − 2 = −3 ≠ 0`. Thói quen `len(s) - 1` thường dành cho bài so sánh **cặp kề nhau**, không dành cho bài duyệt mọi phần tử. **Cách sửa:** `range(len(num))` hoặc `enumerate(num)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng ở vị trí chẵn **so với** tổng ở vị trí lẻ" ⇒ **tổng đan dấu**: gói thành **một** biến `diff` và kiểm tra `== 0`.
- Muốn phân loại theo chẵn/lẻ của **chỉ số**: `enumerate` rồi `i % 2` (hoặc `i & 1`); đừng nhầm với chẵn/lẻ của **giá trị**.
- `s[0::2]` và `s[1::2]` là cách nhanh lấy hai nhóm vị trí chẵn/lẻ.
- Cẩn thận chữ số dạng **ký tự**: luôn `int(ch)` (hoặc `ord(ch) - ord("0")`) trước khi tính toán.
- Có thể loại nhanh: tổng cả chuỗi lẻ ⇒ chắc chắn `False`.
- Đánh chỉ số từ `0` hay `1` không ảnh hưởng đáp án vì điều kiện đối xứng (chỉ hoán đổi vai trò chẵn/lẻ).

## 15. Biến thể của bài

1. **Alternating Digit Sum (2544):** cho số nguyên `n`, chữ số đầu (cao nhất) mang dấu `+`, các chữ số sau đổi dấu luân phiên; trả về tổng. Ví dụ `521 → 5 − 2 + 1 = 4`.
2. **Dấu hiệu chia hết cho 11:** một số chia hết cho `11` khi và chỉ khi tổng đan dấu các chữ số của nó chia hết cho `11` — chính là mở rộng của bài này.
3. **Nhóm theo `i % 3`:** ba tổng ở các chỉ số chia 3 dư `0`, `1`, `2` bằng nhau?
4. **Đếm đoạn con cân bằng:** đếm các đoạn con liên tiếp của `num` mà tổng chẵn = tổng lẻ (tính theo vị trí trong đoạn con). Dùng **tổng đan dấu tiền tố** và đếm các cặp tiền tố bằng nhau.
5. Phiên bản có chỉ số bắt đầu từ `1`: kết quả không đổi (xem mục 10).

In [31]:
from collections import Counter


def alternating_digit_sum(n):
    # chữ số cao nhất mang dấu +, sau đó đổi dấu luân phiên
    digits = [int(c) for c in str(n)]
    return sum(d if i % 2 == 0 else -d for i, d in enumerate(digits))


def divisible_by_11(n):
    # dùng tổng đan dấu (dấu bắt đầu từ chữ số cao nhất; chỉ cần xét chia hết nên dấu đầu không quan trọng)
    return alternating_digit_sum(n) % 11 == 0


def equal_mod3(num):
    sums = [0, 0, 0]
    for i, ch in enumerate(num):
        sums[i % 3] += int(ch)
    return sums[0] == sums[1] == sums[2]


def count_balanced_substrings(num):
    # đoạn [l, r] cân bằng <=> tổng đan dấu toàn cục trên đoạn bằng 0 <=> prefix[r+1] == prefix[l]
    prefix = 0
    seen = Counter({0: 1})
    total = 0
    for i, ch in enumerate(num):
        prefix += int(ch) if i % 2 == 0 else -int(ch)
        total += seen[prefix]            # có bao nhiêu tiền tố trước đó cùng giá trị
        seen[prefix] += 1
    return total


def count_balanced_substrings_brute(num):
    total = 0
    for l in range(len(num)):
        for r in range(l + 1, len(num) + 1):
            part = num[l:r]
            if Solution().isBalanced(part):
                total += 1
    return total


print(alternating_digit_sum(521), alternating_digit_sum(111), alternating_digit_sum(886996))   # 4 1 0
assert [alternating_digit_sum(n) for n in (521, 111, 886996)] == [4, 1, 0]
assert all(divisible_by_11(n) == (n % 11 == 0) for n in range(1, 20000))
assert equal_mod3("123") is False and equal_mod3("333") is True and equal_mod3("000") is True
rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("0123") for _ in range(rng.randint(1, 12)))
    # lưu ý: đoạn con độ dài 1 cũng được tính ("0" cân bằng: 0 == 0); isBalanced của ta xử lý được chuỗi dài 1
    assert count_balanced_substrings(s) == count_balanced_substrings_brute(s), s
print(count_balanced_substrings("1234"), count_balanced_substrings("24123"))

4 1 0
0 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai biến tổng | `even_sum`, `odd_sum` riêng | `O(n)` | `O(1)` | dễ hiểu nhất, bám sát đề |
| 2. Cắt lát bước | `sum(map(int, num[0::2]))` | `O(n)` | `O(n)` | code ngắn, đọc như đề |
| 3. Một biến hiệu | `diff = chẵn − lẻ`, so với `0` | `O(n)` | `O(1)` | **nên nộp**; mở rộng được sang tổng đan dấu |
| 4. Mảng `f[i & 1]` | chỉ số mảng = chẵn/lẻ | `O(n)` | `O(1)` | mẹo bỏ `if`, hợp cho nhiều lớp (`i % 3`) |
| 5. `(-1) ** i` | dấu luân phiên bằng lũy thừa | `O(n)` | `O(1)` | một dòng, đúng kiểu toán học |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `int("7") + int("5")`, `"abcde"[0::2]`, `"abcde"[1::2]`, `Solution().isBalanced("00")`, `Solution().isBalanced("12")`.

**Bài 2 (tính tay).** Cho `"1331"`, `"505"`, `"2002"`. Với mỗi chuỗi, viết tổng chỉ số chẵn, tổng chỉ số lẻ và kết luận.

**Bài 3 (điền TODO).** Hoàn thành `is_balanced_todo(num)` theo Cách 4 (mảng hai ô `f`).

In [32]:
def is_balanced_todo(num):
    # TODO 1: f = [0, 0]
    # TODO 2: duyệt i, ch trong enumerate(num): f[i % 2] += int(ch)
    # TODO 3: trả về f[0] == f[1]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `is_balanced_slices2(num)` theo Cách 2 bằng cắt lát, **không dùng `map`** (dùng generator `sum(int(c) for c in ...)`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `alternating_digit_sum(n)` cho số nguyên `n` (chữ số cao nhất mang dấu `+`): `521 → 4`, `111 → 1`, `886996 → 0`. (b) Viết `divisible_by_11(n)` bằng tổng đan dấu (không dùng `n % 11`), rồi kiểm tra với `n % 11 == 0` trên `1..5000`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `12`, `"ace"`, `"bd"`, `True`, `False`. Bài 3: `f[i % 2]` chọn ô theo chẵn/lẻ.
- **Mức 2 (vừa):** Bài 2: `"1331"`: chẵn `1+3 = 4`, lẻ `3+1 = 4` ⇒ `True`; `"505"`: chẵn `5+5 = 10`, lẻ `0` ⇒ `False`; `"2002"`: `2+0 = 2`, `0+2 = 2` ⇒ `True`. Bài 4: `num[0::2]` và `num[1::2]`.
- **Mức 3 (sâu):** Bài 5(a): chuyển `n` sang chuỗi `str(n)` để lấy từng chữ số theo thứ tự từ cao xuống thấp. Bài 5(b): chỉ cần `tổng_đan_dấu % 11 == 0` (Python cho phần dư không âm khi chia cho số dương, kể cả khi tổng âm).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert int("7") + int("5") == 12
assert "abcde"[0::2] == "ace"
assert "abcde"[1::2] == "bd"
assert Solution().isBalanced("00") is True
assert Solution().isBalanced("12") is False

# Bài 2
for num, expected in [("1331", True), ("505", False), ("2002", True)]:
    even_sum = sum(int(num[i]) for i in range(0, len(num), 2))
    odd_sum = sum(int(num[i]) for i in range(1, len(num), 2))
    assert (even_sum == odd_sum) is expected
    assert Solution().isBalanced(num) is expected


# Bài 3
def is_balanced_todo_ans(num):
    f = [0, 0]
    for i, ch in enumerate(num):
        f[i % 2] += int(ch)
    return f[0] == f[1]


for num, expected in examples:
    assert is_balanced_todo_ans(num) is expected


# Bài 4
def is_balanced_slices2(num):
    return sum(int(c) for c in num[0::2]) == sum(int(c) for c in num[1::2])


for num, expected in examples:
    assert is_balanced_slices2(num) is expected


# Bài 5(a)
def alternating_digit_sum_ans(n):
    total = 0
    sign = 1
    for ch in str(n):
        total += sign * int(ch)
        sign = -sign
    return total


assert alternating_digit_sum_ans(521) == 4
assert alternating_digit_sum_ans(111) == 1
assert alternating_digit_sum_ans(886996) == 0


# Bài 5(b)
def divisible_by_11_ans(n):
    return alternating_digit_sum_ans(n) % 11 == 0


assert all(divisible_by_11_ans(n) == (n % 11 == 0) for n in range(1, 5001))
assert Solution().isBalanced("2112") and not Solution().isBalanced("2113")
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một biến hiệu: chẵn cộng, lẻ trừ)

```python
class Solution:
    def isBalanced(self, num: str) -> bool:
        diff = 0
        for i, ch in enumerate(num):
            if i % 2 == 0:
                diff += int(ch)
            else:
                diff -= int(ch)
        return diff == 0
```

**Checklist trước khi submit**

- [ ] Phân loại theo **chỉ số** `i` (không phải theo giá trị chữ số)?
- [ ] Đã `int(ch)` trước khi cộng (không cộng ký tự, không dùng `ord` thô)?
- [ ] Duyệt đủ **mọi** chữ số (không `range(len(num) - 1)`)?
- [ ] Hàm trả về `bool` (`diff == 0`), không in ra màn hình?
- [ ] Đã thử `"1234" -> False`, `"24123" -> True`, `"00" -> True`, `"12" -> False`, và chuỗi dài 100 chữ số?